### Vanilla
```
input ➡️ LLM ➡️ output
```

In [ ]:
from openai import OpenAI
import dotenv
dotenv.load_dotenv()
import os
import base64
from IPython.display import Markdown, display, Image

##### LLM example 1
This goes to openai servers

In [ ]:
client_oai = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)
response = client_oai.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "Write a short example abstract for a scientific paper. Respond with only the abstract."}]
)
response.choices[0].message.content

##### LLM example 2
This goes to Splinter

In [ ]:
client_splinter = OpenAI(
    base_url="https://llm.science.ai.cam.ac.uk",
    api_key=os.getenv("SPLINTER_API_KEY")
)
response = client_splinter.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Write a short example abstract for a scientific paper. Respond with only the abstract."}]
)
response.choices[0].message.content

##### Other models

We also have other models, for:

- Reasoning
- Vision + Reasoning
- Image Generation
- Image Editing
- Image Recognition
- Text to Speech
- Speech to Text
- Embeddings

See `example-calls.ipynb` for details!

##### OCR example

This goes to a vision model on Splinter

In [ ]:
from agents import ocr
prompt = "Here is some handwritten text and equations in an image. Please read it and convert to markdown, rendering the equations in LaTeX format."
image_path = 'data/imgs/curie.jpg'
display(Image(filename='data/imgs/curie.jpg',height=100))
response = ocr.ocr(image_path, prompt)
display(Markdown(response))

### Chaining
```
input ➡️ LLM1 ➡️ LLM2 ➡️ LLM3 ➡️ output
```

In [ ]:
from agents import ocr
prompt = "Here is some handwritten text and equations in an image. Please read it and convert to markdown, rendering the equations in LaTeX format."
image_path = 'data/imgs/curie.jpg'
response1 = ocr.ocr(image_path, prompt)

In [ ]:
# display(Markdown(response1))

In [ ]:
response2 = client_splinter.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": f"Translate the following to English: {response1}. Respond with only the content, in valid markdown, in english. Use $$ for equations."}]
).choices[0].message.content

In [ ]:
# display(Markdown(response2))

In [ ]:
response3 = client_splinter.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": f"Briefly explain the general context and the topic that these notes are discussing: {response2}. Answer what the area is, who the likely author is, when was it written. Fix likely mistakes introduced by OCR."}]
).choices[0].message.content

In [ ]:
# display(Markdown(response3))

In [ ]:
response4 = client_splinter.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": f"Explain some physics notes line by line to someone with basic physics knowledge. Format it as markdown, explaining all sentences. The context is: {response3}. The notes are: {response2}"}]
).choices[0].message.content

In [ ]:
display(Markdown(response4))

### LangGraph
That was a lot of 'spaghetti' code. Good frameworks exist.

- **LangGraph** (and younger sibling LangChain)
- LlamaIndex
- Microsoft AutoGen

Let's rewrite the above into LangGraph.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from agents import ocr

In [ ]:
from typing import TypedDict

class NotesState(TypedDict, total=False):
    image_path: str
    ocr_markdown: str
    english_markdown: str
    notes_context: str
    line_by_line_explanation: str

def _complete(prompt: str) -> str:
    return client_splinter.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
    ).choices[0].message.content


def ocr_node(state: NotesState) -> NotesState:
    ocr_markdown = ocr.ocr(
        state["image_path"],
        (
            "Here is some handwritten text and equations in an image."
            "Please read it and convert to markdown, rendering the equations in LaTeX format."
        ),
    )
    return {"ocr_markdown": ocr_markdown}


def translate_node(state: NotesState) -> NotesState:
    ocr_markdown = state["ocr_markdown"]

    english_markdown = _complete(
        (
            f"Translate the following to English: {ocr_markdown}. "
            "Respond with only the content, in valid markdown, in english. Use $$ for equations."
        )
    )

    return {"english_markdown": english_markdown}


def context_node(state: NotesState) -> NotesState:
    english_markdown = state["english_markdown"]

    notes_context = _complete(
        (
            "Briefly explain the general context and the topic that these notes are discussing: "
            f"{english_markdown}. "
            "Answer what the area is, who the likely author is, when was it written. "
            "Fix mistakes introduced by OCR."
        )
    )

    return {"notes_context": notes_context}


def explain_node(state: NotesState) -> NotesState:
    english_markdown = state["english_markdown"]
    notes_context = state["notes_context"]

    line_by_line_explanation = _complete(
        (
            "Explain some physics notes line by line to someone with basic physics knowledge. "
            "Format it as markdown, explaining all sentences. "
            f"The context is: {notes_context}. "
            f"The notes are: {english_markdown}"
        )
    )

    return {"line_by_line_explanation": line_by_line_explanation}

In [ ]:
def build_notes_graph(client_splinter):
    graph = StateGraph(NotesState)

    graph.add_node("ocr", ocr_node)
    graph.add_node("translate", translate_node)
    graph.add_node("context", context_node)
    graph.add_node("explain", explain_node)

    graph.add_edge(START, "ocr")
    graph.add_edge("ocr", "translate")
    graph.add_edge("translate", "context")
    graph.add_edge("context", "explain")
    graph.add_edge("explain", END)

    return graph.compile()


app = build_notes_graph(client_splinter)

result = app.invoke({"image_path": "data/imgs/curie.jpg"})

In [ ]:
display(Markdown(result["line_by_line_explanation"]))

### Agentic
```
                  ↗️ LLM2 ➡️ LLM3 ↘️
input ➡️ LLM1 🔀                     🔀 LLM5 ➡️ output
                  ↘️ LLM4 🔄🔄🔄 ↗️
```

In [ ]:
from typing import TypedDict, Dict, Any, Literal, List
import json

from langgraph.graph import StateGraph, END
from agents import code


class AnalysisState(TypedDict, total=False):
    file: str
    question: str
    sandbox_summary: str

    current_script_name: str
    script_history: List[str]
    execution_history: List[Dict[str, Any]]
    findings_history: List[str]

    next_step: Literal["code", "report"]
    decision_reason: str
    report_markdown: str
    iterations: int


def _complete(prompt: str, max_retries: int = 3) -> str:
    last_error = None

    for _ in range(max_retries):
        try:
            resp = client_splinter.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "user", "content": prompt}],
            )
            content = resp.choices[0].message.content
            if content:
                return content
        except Exception as e:
            last_error = e

    return f"(LLM error after {max_retries} attempts: {last_error})"


def _json_preview(obj: Any, max_chars: int = 12000) -> str:
    try:
        text = json.dumps(obj, indent=2, ensure_ascii=True)
    except Exception:
        text = str(obj)

    if len(text) > max_chars:
        return text[:max_chars] + f"\n... (output truncated, length: {len(text)})"
    return text


def _history_block(state: AnalysisState) -> str:
    script_history = state.get("script_history", [])
    execution_history = state.get("execution_history", [])
    findings_history = state.get("findings_history", [])

    parts = []

    if script_history:
        parts.append("Previous scripts:")
        for i, name in enumerate(script_history, start=1):
            parts.append(f"- Iteration {i}: {name}")
        parts.append("")

    if execution_history:
        parts.append("Previous execution results:")
        for i, result in enumerate(execution_history, start=1):
            parts.append(
                f"- Iteration {i}: returncode={result.get('returncode')} "
                f"command={result.get('command')}"
            )
        parts.append("")

    if findings_history:
        parts.append("Previous findings snapshots:")
        for i, finding in enumerate(findings_history, start=1):
            preview = finding[:500]
            if len(finding) > 500:
                preview += f"\n... (output truncated, length: {len(finding)})"
            parts.append(f"Iteration {i} findings preview:\n{preview}\n")

    return "\n".join(parts).strip()


def inspect_node(state: AnalysisState) -> AnalysisState:
    file = state.get("file")
    question = state.get("question")

    if file:
        sandbox_summary = (
            "Sandbox analysis target:\n"
            f"- Focus file: {file}\n\n"
            "Analyze this file primarily. You may use other sandbox files only if helpful.\n\n"
        )
    else:
        inventory = code.run_command("find . -maxdepth 4 | sort", timeout=30)["stdout"]
        sandbox_summary = f"Sandbox contents:\n{inventory}\n\n"

    if question:
        sandbox_summary += f"User objective:\n{question}\n\n"

    return {
        "file": file,
        "question": question,
        "sandbox_summary": sandbox_summary,
        "iterations": 0,
        "script_history": [],
        "execution_history": [],
        "findings_history": [],
    }


def code_node(state: AnalysisState) -> AnalysisState:
    file = state.get("file")
    question = state.get("question", "")
    sandbox_summary = state["sandbox_summary"]
    iterations = state.get("iterations", 0)

    current_script_name = state.get("current_script_name", "")
    script_history = list(state.get("script_history", []))
    execution_history = list(state.get("execution_history", []))
    findings_history = list(state.get("findings_history", []))

    next_script_name = f"analyze_sandbox_v{iterations + 1}.py"

    file_context = ""
    if file:
        file_context = (
            f"Primary target file: {file}\n"
            "Focus the analysis on this file first.\n\n"
        )

    question_block = ""
    if question:
        question_block = f"User objective:\n{question}\n\n"

    previous_script_block = ""
    if current_script_name:
        try:
            previous_script_text = code.read_file(current_script_name, max_bytes=50000)
            previous_script_block = (
                f"Previous script file: {current_script_name}\n\n"
                "Previous script contents:\n"
                f"{previous_script_text}\n\n"
                "Build on this previous script instead of starting over. "
                "Reuse what works, keep the same analysis direction, and extend or fix it.\n\n"
            )
        except Exception as exc:
            previous_script_block = (
                f"Previous script file: {current_script_name}\n"
                f"(unable to read previous script: {exc})\n\n"
            )

    history_block = _history_block(state)
    if history_block:
        history_block = f"{history_block}\n\n"

    last_execution_block = ""
    if execution_history:
        last_result = execution_history[-1]
        last_findings = findings_history[-1] if findings_history else ""
        last_execution_block = (
            "Most recent execution result:\n"
            f"Return code: {last_result.get('returncode')}\n\n"
            f"STDOUT:\n{last_result.get('stdout', '')}\n\n"
            f"STDERR:\n{last_result.get('stderr', '')}\n\n"
            f"Most recent findings.json preview:\n{last_findings}\n\n"
            "Do not discard useful prior work. Improve or extend it.\n\n"
        )

    analysis_script = _complete(
        (
            f"Write a complete Python script named {next_script_name}.\n\n"
            f"{file_context}"
            f"{question_block}"
            f"{sandbox_summary}"
            f"{history_block}"
            f"{previous_script_block}"
            f"{last_execution_block}"
            f"Current iteration: {iterations + 1}\n\n"
            "Requirements:\n"
            "- Run inside ./sandbox.\n"
            "- Analyze the available files in a practical way.\n"
            "- Be defensive and robust.\n"
            "- No manual input.\n"
            "- Prefer standard library.\n"
            "- If optional libraries are unavailable, degrade gracefully.\n"
            "- Build on previous analyses rather than restarting from scratch.\n"
            "- It is fine to read outputs from previous iterations if useful.\n"
            "- Write structured results to findings.json.\n"
            "- Print a short summary to stdout.\n"
            "- Return only valid Python code, no markdown fences."
        )
    )

    code.write_file(next_script_name, analysis_script)

    script_history.append(next_script_name)

    return {
        "file": file,
        "question": question,
        "current_script_name": next_script_name,
        "script_history": script_history,
        "iterations": iterations + 1,
    }


def execute_node(state: AnalysisState) -> AnalysisState:
    file = state.get("file")
    question = state.get("question")
    current_script_name = state["current_script_name"]

    execution_history = list(state.get("execution_history", []))
    findings_history = list(state.get("findings_history", []))

    execution_result = code.run_command(f"python {current_script_name}", timeout=120)

    try:
        findings_json = code.read_file("findings.json", max_bytes=30000)
    except Exception as exc:
        findings_json = f"(unable to read findings.json: {exc})"

    execution_history.append(execution_result)
    findings_history.append(findings_json)

    return {
        "file": file,
        "question": question,
        "execution_history": execution_history,
        "findings_history": findings_history,
    }


def decide_node(state: AnalysisState) -> AnalysisState:
    file = state.get("file")
    question = state.get("question", "")
    sandbox_summary = state["sandbox_summary"]
    iterations = state.get("iterations", 0)
    current_script_name = state.get("current_script_name", "")

    execution_history = state.get("execution_history", [])
    findings_history = state.get("findings_history", [])

    latest_execution = execution_history[-1] if execution_history else {}
    latest_findings = findings_history[-1] if findings_history else ""

    file_context = f"Primary target file: {file}\n\n" if file else ""
    question_block = f"User objective:\n{question}\n\n" if question else ""
    history_block = _history_block(state)
    if history_block:
        history_block += "\n\n"

    decision_text = _complete(
        (
            "Decide whether to run another code-writing iteration or write the final report.\n\n"
            "Respond in exactly this format:\n"
            "NEXT_STEP: code\n"
            "REASON: <short reason>\n\n"
            "or\n\n"
            "NEXT_STEP: report\n"
            "REASON: <short reason>\n\n"
            "Choose code if:\n"
            "- the latest script failed\n"
            "- findings.json is missing, weak, or incomplete\n"
            "- another iteration would materially improve the analysis\n\n"
            "Choose report if:\n"
            "- the accumulated analysis is good enough to summarize\n"
            "- more iteration is unlikely to help much\n\n"
            "Prefer not to loop forever, but aim for a couple good iterations.\n\n"
            f"{file_context}"
            f"{question_block}"
            f"{sandbox_summary}"
            f"{history_block}"
            f"Current script: {current_script_name}\n\n"
            f"Iteration count: {iterations}\n\n"
            "Latest execution result:\n"
            f"Return code: {latest_execution.get('returncode')}\n\n"
            f"STDOUT:\n{latest_execution.get('stdout', '')}\n\n"
            f"STDERR:\n{latest_execution.get('stderr', '')}\n\n"
            f"Latest findings.json:\n{latest_findings}\n"
        )
    ) or ""

    next_step = "report"
    reason = "Analysis appears sufficient."

    for line in decision_text.splitlines():
        if line.startswith("NEXT_STEP:"):
            value = line.split(":", 1)[1].strip()
            if value in {"code", "report"}:
                next_step = value
        elif line.startswith("REASON:"):
            reason = line.split(":", 1)[1].strip()

    if iterations >= 5 and next_step == "code":
        next_step = "report"
        reason = "Maximum iterations reached; producing best-effort report."

    return {
        "file": file,
        "question": question,
        "next_step": next_step,
        "decision_reason": reason,
    }


def report_node(state: AnalysisState) -> AnalysisState:
    file = state.get("file")
    question = state.get("question", "")
    sandbox_summary = state["sandbox_summary"]
    decision_reason = state.get("decision_reason", "")
    current_script_name = state.get("current_script_name", "")

    script_history = state.get("script_history", [])
    execution_history = state.get("execution_history", [])
    findings_history = state.get("findings_history", [])

    latest_execution = execution_history[-1] if execution_history else {}
    latest_findings = findings_history[-1] if findings_history else ""
    history_block = _history_block(state)

    file_context = f"Primary target file: {file}\n\n" if file else ""
    question_block = f"User objective:\n{question}\n\n" if question else ""

    report_markdown = _complete(
        (
            "Write a short paper-like markdown report based on the sandbox analysis.\n\n"
            f"{file_context}"
            f"{question_block}"
            f"{sandbox_summary}"
            f"Final decision note: {decision_reason}\n\n"
            f"Final script used: {current_script_name}\n"
            f"Total iterations: {len(script_history)}\n\n"
            f"Analysis history:\n{history_block}\n\n"
            "Latest execution result:\n"
            f"Return code: {latest_execution.get('returncode')}\n\n"
            f"STDOUT:\n{latest_execution.get('stdout', '')}\n\n"
            f"STDERR:\n{latest_execution.get('stderr', '')}\n\n"
            f"Latest findings.json:\n{latest_findings}\n\n"
            "Use these sections:\n"
            "- Title\n"
            "- Abstract\n"
            "- Files Examined\n"
            "- Method\n"
            "- Findings\n"
            "- Limitations\n"
            "- Conclusion\n\n"
            "Keep it short, concrete, and honest about uncertainty. "
            "If later iterations refined earlier results, reflect the accumulated analysis rather than only the last run. "
            "Format it as markdown, use $$ $$."
        )
    )

    return {
        "file": file,
        "question": question,
        "report_markdown": report_markdown,
    }


def route_after_decision(state: AnalysisState):
    return state["next_step"]


In [ ]:
workflow = StateGraph(AnalysisState)

workflow.add_node("inspect", inspect_node)
workflow.add_node("code", code_node)
workflow.add_node("execute", execute_node)
workflow.add_node("decide", decide_node)
workflow.add_node("report", report_node)

workflow.set_entry_point("inspect")
workflow.add_edge("inspect", "code")
workflow.add_edge("code", "execute")
workflow.add_edge("execute", "decide")
workflow.add_conditional_edges(
    "decide",
    route_after_decision,
    {
        "code": "code",
        "report": "report",
    },
)
workflow.add_edge("report", END)

analysis_graph = workflow.compile()

inputs = {
    "file": "audience.csv",
    "question": "Analyse this to find unique people, be careful to consider near misses as people change names and one column can be either id or id@cam.ac.uk, sometimes you might have to judge manually. After that, report various important summary stats about what kind of people engage with us, and which engagement routes are popular with which departments and career stages. You can write python code, and execute commands.",
}

for step in analysis_graph.stream(inputs):
    print(next(iter(step)))

result = analysis_graph.invoke(inputs)
display(Markdown(result["report_markdown"]))

### Your agents go here

Define your state

In [ ]:
# class YourState(TypedDict, total=False):
#     file: str
#     question: str
#     ...

Define your nodes

In [ ]:
# def ocr_node(state: YourState) -> YourState:
#     ocr_markdown = ocr.ocr(
#         state["image_path"],
#         (
#             "Here is some handwritten text and equations in an image. "
#             "Please read it and convert to markdown, rendering the equations in LaTeX format."
#         ),
#     )
#     return {"ocr_markdown": ocr_markdown}
#
# ...

Draw edges

In [ ]:
# workflow = StateGraph(YourState)

# workflow.add_node("inspect", inspect_node)
# workflow.add_node("code", code_node)
# ...

# workflow.set_entry_point("inspect")
# workflow.add_edge("inspect", "code")
# ...
# workflow.add_conditional_edges(
#     "decide",
#     route_after_decision,
#     {
#         "code": "code",
#         "report": "todo",
#     },
# )
# workflow.add_edge("todo", END)

Run it!

In [ ]:
# your_graph = workflow.compile()

# for step in your_graph.stream({
#     ...
# }):
#     print(next(iter(step)))

# result = your_graph.invoke({
#     ...
# })
# display(Markdown(result["report_markdown"]))